# Graph-Based Machine Learning for Distribution Network Reconfiguration
## Physics-Informed GNNs, Graph Transformers & Meta-Learning on the IEEE 33-Bus System

**Research Question:** Can Physics-Informed Graph Neural Networks, Graph Transformers, and Meta-Learning architectures learn near-optimal distribution network reconfiguration policies that generalise across unseen load and DER conditions?

**Primary Deployment Goal:** Near-optimal performance · Fast inference · Generalisation to unseen operating conditions

---
*Suitable for: IEEE Transactions on Smart Grid · IEEE Transactions on Power Systems · Applied Energy · Electric Power Systems Research*


## 0 · Imports, Reproducibility & Device Configuration

In [ ]:

import os, sys, time, random, warnings, copy, json
from pathlib import Path
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
import numpy as np
np.random.seed(SEED)

import torch
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")

import pandas as pd
import scipy.stats as st
from scipy.stats import wilcoxon, ttest_rel
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix)
from sklearn.preprocessing import StandardScaler
import networkx as nx

import pandapower as pp
import pandapower.networks as pn

import torch_geometric
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import (GCNConv, SAGEConv, GATConv,
                                  TransformerConv, global_mean_pool,
                                  global_max_pool)
from torch_geometric.utils import to_networkx
from torch_geometric.loader import DataLoader as PyGLoader

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

DATA_DIR   = Path('data');   DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR  = Path('models'); MODEL_DIR.mkdir(exist_ok=True)
FIG_DIR    = Path('figures');FIG_DIR.mkdir(exist_ok=True)

print("All imports successful.")
print(f"Torch Geometric: {torch_geometric.__version__}")


---
## Section 1 · IEEE 33-Bus Distribution Network

The IEEE 33-bus radial feeder is the canonical benchmark for distribution network reconfiguration research.  It contains **33 buses**, **32 normally-closed sectionalising switches**, and **5 normally-open tie switches**, totalling 37 branches.  The nominal voltage is **12.66 kV** with a total load of approximately **3.715 MW + j2.300 MVAr**.


In [ ]:

def build_ieee33():
    """Construct the IEEE 33-bus radial distribution network."""
    net = pp.create_empty_network(f_hz=60, sn_mva=10)
    vn = 12.66
    for i in range(33):
        pp.create_bus(net, vn_kv=vn, name=f'Bus {i+1}', index=i)
    pp.create_ext_grid(net, bus=0, vm_pu=1.0, va_degree=0.0, name='Substation')

    branch_data = [
        (0,1,0.0922,0.0470,0.400),(1,2,0.4930,0.2511,0.400),
        (2,3,0.3660,0.1864,0.400),(3,4,0.3811,0.1941,0.400),
        (4,5,0.8190,0.7070,0.400),(5,6,0.1872,0.6188,0.400),
        (6,7,0.7114,0.2351,0.400),(7,8,1.0300,0.7400,0.400),
        (8,9,1.0440,0.7400,0.400),(9,10,0.1966,0.0650,0.400),
        (10,11,0.3744,0.1238,0.400),(11,12,1.4680,1.1550,0.400),
        (12,13,0.5416,0.7129,0.400),(13,14,0.5910,0.5260,0.400),
        (14,15,0.7463,0.5450,0.400),(15,16,1.2890,1.7210,0.400),
        (16,17,0.7320,0.5740,0.400),(17,18,0.1640,0.1565,0.400),
        (18,19,1.5042,1.3554,0.400),(19,20,0.4095,0.4784,0.400),
        (20,21,0.7089,0.9373,0.400),(21,22,0.4512,0.3083,0.400),
        (1,23,0.8980,0.7091,0.400),(23,24,0.8960,0.7011,0.400),
        (5,25,0.2030,0.1034,0.400),(25,26,0.2842,0.1447,0.400),
        (26,27,1.0590,0.9337,0.400),(27,28,0.8042,0.7006,0.400),
        (28,29,0.5075,0.2585,0.400),(29,30,0.9744,0.9630,0.400),
        (30,31,0.3105,0.3619,0.400),(31,32,0.3410,0.5302,0.400),
        (8,20,2.0000,2.0000,0.400),(11,14,2.0000,2.0000,0.400),
        (17,32,0.5000,0.5000,0.400),(24,28,0.5000,0.5000,0.400),
        (6,15,2.0000,2.0000,0.400),
    ]
    line_indices = []
    for i, (fb, tb, r, x, max_i) in enumerate(branch_data):
        idx = pp.create_line_from_parameters(
            net, from_bus=fb, to_bus=tb, length_km=1.0,
            r_ohm_per_km=r, x_ohm_per_km=x, c_nf_per_km=0,
            max_i_ka=max_i, name=f'Line {i+1}', in_service=True)
        line_indices.append(idx)

    load_data = [
        (1,100,60),(2,90,40),(3,120,80),(4,60,30),(5,60,20),
        (6,200,100),(7,200,100),(8,60,20),(9,60,20),(10,45,30),
        (11,60,35),(12,60,35),(13,120,80),(14,60,10),(15,60,20),
        (16,60,20),(17,90,40),(18,90,40),(19,90,40),(20,90,40),
        (21,90,40),(22,90,40),(23,420,200),(24,420,200),(25,60,25),
        (26,60,25),(27,60,20),(28,120,70),(29,200,600),(30,150,70),
        (31,210,100),(32,60,40),
    ]
    for bus, p, q in load_data:
        pp.create_load(net, bus=bus, p_mw=p/1000, q_mvar=q/1000, name=f'Load B{bus+1}')

    switch_indices = []
    for li in line_indices[-5:]:
        row = net.line.loc[li]
        sw = pp.create_switch(net, bus=int(row.from_bus), element=li, et='l',
                              closed=False, name=f'SW_tie_{li}')
        switch_indices.append(sw)
        net.line.at[li, 'in_service'] = False
    return net, switch_indices

net, TIE_SWITCHES = build_ieee33()
pp.runpp(net, algorithm='nr', numba=False)
print("Power flow converged:", net.converged)
print(f"Buses: {len(net.bus)}  |  Lines: {len(net.line)}  |  Loads: {len(net.load)}")
print(f"Total P load: {net.load.p_mw.sum()*1000:.1f} kW")
print(f"Total Q load: {net.load.q_mvar.sum()*1000:.1f} kVAr")
print(f"System losses: {net.res_line.pl_mw.sum()*1000:.2f} kW")
print(f"Min voltage  : {net.res_bus.vm_pu.min():.4f} pu  (Bus {net.res_bus.vm_pu.idxmin()+1})")


### 1.1 · Interactive Feeder Topology Visualisation

In [ ]:

def build_nx_graph(net):
    """Convert pandapower network to a NetworkX graph with attributes."""
    G = nx.Graph()
    for idx, row in net.bus.iterrows():
        G.add_node(idx, name=row['name'], vn_kv=row['vn_kv'])
    for idx, row in net.line.iterrows():
        G.add_edge(int(row.from_bus), int(row.to_bus), line_idx=idx,
                   r=row.r_ohm_per_km, x=row.x_ohm_per_km,
                   in_service=row.in_service, is_tie=(idx >= 32))
    return G

G = build_nx_graph(net)
pos = nx.spring_layout(G, seed=SEED, k=2.5)
node_x, node_y, node_text, node_color = [], [], [], []
for n in G.nodes():
    x, y = pos[n]
    node_x.append(x); node_y.append(y)
    node_text.append(f'Bus {n+1}')
    node_color.append('red' if n == 0 else 'steelblue')

edge_traces = []
for u, v, d in G.edges(data=True):
    x0,y0 = pos[u]; x1,y1 = pos[v]
    color = 'orange' if d['is_tie'] else ('lightgray' if not d['in_service'] else '#555')
    dash = 'dash' if d['is_tie'] else 'solid'
    edge_traces.append(go.Scatter(x=[x0,x1,None], y=[y0,y1,None], mode='lines',
                                  line=dict(color=color, width=2.5 if d['is_tie'] else 1.8, dash=dash),
                                  hoverinfo='none', showlegend=False))

fig = go.Figure(data=edge_traces + [go.Scatter(
    x=node_x, y=node_y, mode='markers+text',
    marker=dict(size=14, color=node_color, line=dict(width=1.5, color='white')),
    text=node_text, textposition='top center', textfont=dict(size=8),
    hovertext=[f'Bus {n+1}' for n in G.nodes()], hoverinfo='text', name='Buses')])
fig.update_layout(title='IEEE 33-Bus Distribution Feeder<br><sup>Blue=load bus | Red=slack | Orange dashed=tie switch (N.O.)</sup>',
                  showlegend=False, hovermode='closest', margin=dict(b=20,l=5,r=5,t=60),
                  xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                  yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                  height=550, paper_bgcolor='white', plot_bgcolor='white')
fig.show()
fig.write_html(str(FIG_DIR/'fig1_feeder_topology.html'))
print("Figure 1 saved.")


---
## Section 2 · Scenario Generation Pipeline

We generate **10 000 synthetic operating scenarios** by randomising load variability, solar PV, wind generation, battery storage, and EV charging demand. Each scenario is stored as a feature vector and later converted into a PyTorch Geometric graph.


In [ ]:

N_SCENARIOS = 10_000
N_BUSES = 33
SOLAR_BUSES = [3, 7, 13, 19, 24, 29]
WIND_BUSES = [5, 16, 27]
BATTERY_BUSES = [8, 14, 21]
EV_BUSES = [4, 10, 17, 22, 30]
BASE_P = net.load.p_mw.values.copy()
BASE_Q = net.load.q_mvar.values.copy()
SCENARIO_CACHE = DATA_DIR / 'scenarios.npy'

def generate_scenarios(n=N_SCENARIOS, seed=SEED):
    rng = np.random.default_rng(seed)
    records = []
    for sid in range(n):
        lm = rng.lognormal(mean=0.0, sigma=0.2, size=len(BASE_P))
        lm = np.clip(lm, 0.5, 1.5)
        P_load = BASE_P * lm
        Q_load = BASE_Q * lm
        solar_irr = rng.beta(2, 5, size=N_BUSES)
        p_solar = np.zeros(N_BUSES)
        for b in SOLAR_BUSES:
            cap = rng.uniform(0.1, 0.5)
            p_solar[b] = solar_irr[b] * cap
        p_wind = np.zeros(N_BUSES)
        for b in WIND_BUSES:
            speed = np.clip(rng.weibull(2.0) * 8, 0, 25)
            if speed < 3.5: pw = 0.0
            elif speed < 14: pw = 0.15 * (speed/14)**3
            elif speed < 25: pw = 0.15
            else: pw = 0.0
            p_wind[b] = pw
        p_batt = np.zeros(N_BUSES); soc = np.zeros(N_BUSES)
        for b in BATTERY_BUSES:
            soc[b] = rng.uniform(0.1, 0.9)
            max_kw = rng.uniform(0.05, 0.2)
            p_batt[b] = rng.choice([-1, 1]) * max_kw * rng.uniform(0.3, 1.0)
        p_ev = np.zeros(N_BUSES)
        for b in EV_BUSES:
            if rng.random() < 0.6:
                p_ev[b] = rng.uniform(0.02, 0.15)
        records.append({'scenario_id': sid, 'P_load_mw': P_load, 'Q_load_mvar': Q_load,
                        'P_solar_mw': p_solar, 'P_wind_mw': p_wind,
                        'P_batt_mw': p_batt, 'SoC': soc, 'P_ev_mw': p_ev})
    return records

if SCENARIO_CACHE.exists():
    print(f"Loading scenarios from {SCENARIO_CACHE} …")
    scenarios = np.load(str(SCENARIO_CACHE), allow_pickle=True).tolist()
    if len(scenarios) != N_SCENARIOS:
        print(f"Cached scenarios contain {len(scenarios)} records; regenerating {N_SCENARIOS} scenarios …")
        t0 = time.time()
        scenarios = generate_scenarios()
        np.save(str(SCENARIO_CACHE), np.array(scenarios, dtype=object), allow_pickle=True)
        print(f"Regenerated and saved scenarios in {time.time()-t0:.1f}s")
    else:
        print(f"Loaded {len(scenarios)} scenarios")
else:
    print("Generating 10 000 scenarios …")
    t0 = time.time()
    scenarios = generate_scenarios()
    np.save(str(SCENARIO_CACHE), np.array(scenarios, dtype=object), allow_pickle=True)
    print(f"Done in {time.time()-t0:.1f}s  |  {len(scenarios)} scenarios")
    print(f"Saved scenarios to {SCENARIO_CACHE}")

print(f"Avg total P load : {np.mean([s['P_load_mw'].sum() for s in scenarios])*1000:.1f} kW")
print(f"Avg total Solar  : {np.mean([s['P_solar_mw'].sum() for s in scenarios])*1000:.1f} kW")


---
## Section 3 · Optimal Label Generation

For each scenario we apply scenario loads/DERs, run AC power flow, evaluate a multi-objective score, search switch configurations, and store graph-feature/label pairs.


In [ ]:

W_LOSS = 1.0
W_VDEV = 50.0
W_OVERLD = 100.0
V_MIN, V_MAX = 0.95, 1.05
TIE_LINE_IDX = list(range(32, 37))

def apply_scenario(net_ref, scenario):
    net2 = copy.deepcopy(net_ref)
    for i, idx in enumerate(net2.load.index):
        net2.load.at[idx, 'p_mw'] = float(scenario['P_load_mw'][i])
        net2.load.at[idx, 'q_mvar'] = float(scenario['Q_load_mvar'][i])
    for key, buses, sign in [('P_solar_mw', SOLAR_BUSES, -1), ('P_wind_mw', WIND_BUSES, -1),
                             ('P_batt_mw', BATTERY_BUSES, -1), ('P_ev_mw', EV_BUSES, +1)]:
        arr = scenario[key]
        for b in buses:
            load_rows = net2.load[net2.load.bus == b].index
            if len(load_rows):
                net2.load.at[load_rows[0], 'p_mw'] += sign * float(arr[b])
    net2.load['p_mw'] = net2.load['p_mw'].clip(lower=0.001)
    return net2

def compute_objective(net2):
    try:
        pp.runpp(net2, algorithm='nr', numba=False, max_iteration=50, tolerance_mva=1e-6)
        if not net2.converged:
            return 1e9, False
    except Exception:
        return 1e9, False
    loss = net2.res_line.pl_mw.sum()
    vdev = ((net2.res_bus.vm_pu - 1.0)**2).sum()
    overl = (np.maximum(net2.res_line.loading_percent / 100 - 1.0, 0)**2).sum()
    return float(W_LOSS*loss + W_VDEV*vdev + W_OVERLD*overl), True

def set_switch_config(net2, config_5bit):
    for bit, li in enumerate(TIE_LINE_IDX):
        net2.line.at[li, 'in_service'] = bool((config_5bit >> bit) & 1)

def find_optimal_config(net_scenario, n_configs=32):
    best_obj, best_config = 1e9, 0
    for cfg in range(n_configs):
        net2 = copy.deepcopy(net_scenario)
        set_switch_config(net2, cfg)
        obj, conv = compute_objective(net2)
        if conv and obj < best_obj:
            best_obj, best_config = obj, cfg
    return best_config, best_obj

def extract_bus_features(net2):
    feats = np.zeros((N_BUSES, 7), dtype=np.float32)
    feats[:,0] = net2.res_bus.vm_pu.values.astype(np.float32)
    feats[:,1] = net2.res_bus.va_degree.values.astype(np.float32)
    for _, row in net2.load.iterrows():
        b = int(row.bus)
        feats[b,2] += float(row.p_mw)
        feats[b,3] += float(row.q_mvar)
    return feats

def extract_line_features(net2):
    n = len(net2.line)
    feats = np.zeros((n, 4), dtype=np.float32)
    feats[:,0] = net2.line.r_ohm_per_km.values.astype(np.float32)
    feats[:,1] = net2.line.x_ohm_per_km.values.astype(np.float32)
    feats[:,2] = net2.line.in_service.values.astype(np.float32)
    feats[:,3] = net2.res_line.loading_percent.values.astype(np.float32) / 100
    return feats


In [ ]:

def process_scenario(args):
    sid, scenario = args
    net_s = apply_scenario(net, scenario)
    set_switch_config(net_s, 0)
    _, conv = compute_objective(net_s)
    if not conv:
        return None
    node_f = extract_bus_features(net_s)
    opt_cfg, opt_obj = find_optimal_config(net_s)
    net_opt = copy.deepcopy(net_s)
    set_switch_config(net_opt, opt_cfg)
    pp.runpp(net_opt, algorithm='nr', numba=False, max_iteration=50)
    edge_f = extract_line_features(net_opt)
    sw_label = np.ones(37, dtype=np.float32)
    for bit, li in enumerate(TIE_LINE_IDX):
        sw_label[li] = float((opt_cfg >> bit) & 1)
    return {'sid': sid, 'node_f': node_f, 'edge_f': edge_f,
            'sw_label': sw_label, 'opt_cfg': opt_cfg, 'opt_obj': opt_obj}

DATASET_CACHE = DATA_DIR / 'dataset_raw.npy'

if DATASET_CACHE.exists():
    print(f"Loading labeled dataset from {DATASET_CACHE} …")
    dataset_raw = np.load(str(DATASET_CACHE), allow_pickle=True).tolist()
    print(f"Loaded {len(dataset_raw)} valid scenarios. Continuing with PyG conversion/training.")
else:
    print("Generating labels for 10 000 scenarios …")
    print("(Each scenario: 32 power flows. May take several minutes.)")
    dataset_raw = []
    BATCH_SIZE = 200
    t0 = time.time()
    for batch_start in range(0, N_SCENARIOS, BATCH_SIZE):
        batch = [(sid, scenarios[sid]) for sid in range(batch_start, min(batch_start+BATCH_SIZE, N_SCENARIOS))]
        for args in batch:
            result = process_scenario(args)
            if result is not None:
                dataset_raw.append(result)
        elapsed = time.time() - t0
        done = min(batch_start + BATCH_SIZE, N_SCENARIOS)
        rate = done / max(elapsed, 1e-9)
        eta = (N_SCENARIOS - done) / max(rate, 1)
        print(f"  {done}/{N_SCENARIOS}  |  {elapsed:.0f}s elapsed  |  ETA {eta:.0f}s", end='\r')
    print(f"\nDataset: {len(dataset_raw)} valid scenarios  ({time.time()-t0:.0f}s total)")
    np.save(str(DATASET_CACHE), np.array(dataset_raw, dtype=object), allow_pickle=True)
    print(f"Saved labeled dataset to {DATASET_CACHE}")


---
## Section 4 · PyTorch Geometric Graph Construction

Each scenario is encoded as a graph `G = (V, E, X_v, X_e)` with node features, bidirectional edge features, topology adjacency, and binary switch labels.


In [ ]:

def build_edge_index(net):
    src, dst = [], []
    for _, row in net.line.iterrows():
        u, v = int(row.from_bus), int(row.to_bus)
        src += [u, v]
        dst += [v, u]
    return torch.tensor([src, dst], dtype=torch.long)

EDGE_INDEX = build_edge_index(net)

def scenario_to_pyg(record):
    nf = record['node_f'].copy()
    ef = record['edge_f']
    ef2 = np.concatenate([ef, ef], axis=0)
    return Data(x=torch.tensor(nf, dtype=torch.float32), edge_index=EDGE_INDEX.clone(),
                edge_attr=torch.tensor(ef2, dtype=torch.float32),
                y=torch.tensor(record['sw_label'], dtype=torch.float32),
                opt_cfg=torch.tensor([record['opt_cfg']], dtype=torch.long),
                opt_obj=torch.tensor([record['opt_obj']], dtype=torch.float32))

print("Converting to PyG Data objects …")
pyg_dataset = [scenario_to_pyg(r) for r in dataset_raw]
print(f"PyG dataset: {len(pyg_dataset)} graphs")
print(f"Example graph: {pyg_dataset[0]}")
all_nf = np.stack([r['node_f'] for r in dataset_raw])
NF_MEAN = torch.tensor(all_nf.mean(axis=(0,1)), dtype=torch.float32)
NF_STD = torch.tensor(all_nf.std(axis=(0,1)) + 1e-6, dtype=torch.float32)
for g in pyg_dataset:
    g.x = (g.x - NF_MEAN) / NF_STD
torch.save(NF_MEAN, str(DATA_DIR/'nf_mean.pt'))
torch.save(NF_STD, str(DATA_DIR/'nf_std.pt'))


In [ ]:

N = len(pyg_dataset)
n_train = int(0.70 * N)
n_val = int(0.15 * N)
n_test = N - n_train - n_val
idx = np.random.default_rng(SEED).permutation(N)
train_data = [pyg_dataset[i] for i in idx[:n_train]]
val_data = [pyg_dataset[i] for i in idx[n_train:n_train+n_val]]
test_data = [pyg_dataset[i] for i in idx[n_train+n_val:]]
BATCH = 64
train_loader = PyGLoader(train_data, batch_size=BATCH, shuffle=True)
val_loader = PyGLoader(val_data, batch_size=BATCH, shuffle=False)
test_loader = PyGLoader(test_data, batch_size=BATCH, shuffle=False)
print(f"Train: {len(train_data)}  Val: {len(val_data)}  Test: {len(test_data)}")
print(f"Batches per epoch: {len(train_loader)}")


In [ ]:

sample = pyg_dataset[0]
G_nx = to_networkx(sample, to_undirected=True)
pos = nx.spring_layout(G_nx, seed=SEED, k=2.0)
vm_vals = dataset_raw[0]['node_f'][:, 0]
fig2 = go.Figure()
for u, v in G_nx.edges():
    x0,y0 = pos[u]; x1,y1 = pos[v]
    tie_pairs = [(int(net.line.loc[li,'from_bus']), int(net.line.loc[li,'to_bus'])) for li in TIE_LINE_IDX]
    is_tie = (u, v) in tie_pairs or (v, u) in tie_pairs
    fig2.add_trace(go.Scatter(x=[x0,x1,None], y=[y0,y1,None], mode='lines',
                              line=dict(color='orange' if is_tie else '#aaa', width=2),
                              showlegend=False, hoverinfo='none'))
fig2.add_trace(go.Scatter(x=[pos[n][0] for n in G_nx.nodes()], y=[pos[n][1] for n in G_nx.nodes()],
                          mode='markers+text',
                          marker=dict(size=16, color=vm_vals, colorscale='RdYlGn', showscale=True,
                                      colorbar=dict(title='Vm (pu)'), line=dict(width=1,color='black')),
                          text=[f'B{n+1}' for n in G_nx.nodes()], textposition='top center', textfont=dict(size=7),
                          hovertext=[f'Bus {n+1}<br>Vm={vm_vals[n]:.4f} pu' for n in G_nx.nodes()],
                          hoverinfo='text', name='Buses'))
fig2.update_layout(title='Graph Representation – Node Voltage Coloring (Scenario #0)', height=520,
                   xaxis=dict(showgrid=False,zeroline=False,showticklabels=False),
                   yaxis=dict(showgrid=False,zeroline=False,showticklabels=False),
                   paper_bgcolor='white', plot_bgcolor='white')
fig2.show()
fig2.write_html(str(FIG_DIR/'fig2_graph_voltage.html'))
print("Figure 2 saved.")


---
## Section 5 · Baseline GNN Models

We implement three canonical GNN architectures as baselines: **GCN**, **GraphSAGE**, and **GAT**. All models use graph-level readout and predict a 37-switch binary vector.


In [ ]:

NODE_DIM = 7
EDGE_DIM = 4
HIDDEN = 128
N_LAYERS = 4
N_HEADS = 4
DROPOUT = 0.15
N_SWITCH = 37
EPOCHS_BASE = 60
LR = 3e-4

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        loss = criterion(out, batch.y.view(-1, N_SWITCH))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)

@torch.no_grad()
def eval_model(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    for batch in loader:
        batch = batch.to(device)
        out = model(batch)
        loss = criterion(out, batch.y.view(-1, N_SWITCH))
        total_loss += loss.item() * batch.num_graphs
        all_preds.append(torch.sigmoid(out).cpu().numpy())
        all_labels.append(batch.y.view(-1, N_SWITCH).cpu().numpy())
    preds = np.vstack(all_preds); labels = np.vstack(all_labels)
    bin_p = (preds > 0.5).astype(int)
    acc = accuracy_score(labels.flatten(), bin_p.flatten())
    f1 = f1_score(labels.flatten(), bin_p.flatten(), zero_division=0)
    try: auc = roc_auc_score(labels.flatten(), preds.flatten())
    except Exception: auc = 0.5
    return total_loss / len(loader.dataset), acc, f1, auc

def run_training(model, model_name, epochs=EPOCHS_BASE, device=DEVICE, lr=LR):
    model = model.to(device)
    opt = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch = CosineAnnealingLR(opt, T_max=epochs, eta_min=lr*0.05)
    crit = nn.BCEWithLogitsLoss()
    history = {'train_loss':[], 'val_loss':[], 'val_acc':[], 'val_f1':[]}
    best_val, best_state = 1e9, None
    for epoch in range(1, epochs+1):
        tl = train_epoch(model, train_loader, opt, crit, device)
        vl, va, vf1, vauc = eval_model(model, val_loader, crit, device)
        sch.step()
        history['train_loss'].append(tl); history['val_loss'].append(vl)
        history['val_acc'].append(va); history['val_f1'].append(vf1)
        if vl < best_val:
            best_val, best_state = vl, copy.deepcopy(model.state_dict())
        if epoch % 10 == 0:
            print(f"[{model_name}] Ep {epoch:03d} | TL {tl:.4f} VL {vl:.4f} Acc {va:.4f} F1 {vf1:.4f} AUC {vauc:.4f}")
    model.load_state_dict(best_state)
    torch.save(best_state, str(MODEL_DIR/f'{model_name}.pt'))
    print(f"[{model_name}] Best val loss: {best_val:.4f}")
    return model, history


In [ ]:

class GCN(nn.Module):
    def __init__(self, in_dim=NODE_DIM, hidden=HIDDEN, out_dim=N_SWITCH, n_layers=N_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, hidden)
        self.convs = nn.ModuleList([GCNConv(hidden, hidden) for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(hidden*2, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, out_dim))
    def forward(self, data):
        x, ei, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.input_proj(x))
        for conv, norm in zip(self.convs, self.norms):
            x = norm(F.relu(conv(x, ei)) + x)
            x = self.dropout(x)
        return self.head(torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=-1))

model_gcn = GCN()
print(f"GCN parameters: {sum(p.numel() for p in model_gcn.parameters()):,}")
model_gcn, hist_gcn = run_training(model_gcn, 'GCN')


In [ ]:

class GraphSAGE(nn.Module):
    def __init__(self, in_dim=NODE_DIM, hidden=HIDDEN, out_dim=N_SWITCH, n_layers=N_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, hidden)
        self.convs = nn.ModuleList([SAGEConv(hidden, hidden) for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(hidden*2, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, out_dim))
    def forward(self, data):
        x, ei, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.input_proj(x))
        for conv, norm in zip(self.convs, self.norms):
            x = norm(F.relu(conv(x, ei)) + x)
            x = self.dropout(x)
        return self.head(torch.cat([global_mean_pool(x,batch), global_max_pool(x,batch)], -1))

model_sage = GraphSAGE()
print(f"GraphSAGE parameters: {sum(p.numel() for p in model_sage.parameters()):,}")
model_sage, hist_sage = run_training(model_sage, 'GraphSAGE')


In [ ]:

class GAT(nn.Module):
    def __init__(self, in_dim=NODE_DIM, hidden=HIDDEN, out_dim=N_SWITCH, n_layers=N_LAYERS, heads=N_HEADS, dropout=DROPOUT):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, hidden)
        self.convs = nn.ModuleList([GATConv(hidden, hidden//heads, heads=heads, dropout=dropout, concat=True) for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(hidden*2, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, out_dim))
        self.last_attn = None
    def forward(self, data, return_attn=False):
        x, ei, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.input_proj(x))
        for i, (conv, norm) in enumerate(zip(self.convs, self.norms)):
            if return_attn and i == len(self.convs)-1:
                x_new, (_, attn) = conv(x, ei, return_attention_weights=True)
                self.last_attn = attn.detach().cpu()
            else:
                x_new = conv(x, ei)
            x = norm(F.relu(x_new) + x)
            x = self.dropout(x)
        return self.head(torch.cat([global_mean_pool(x,batch), global_max_pool(x,batch)], -1))

model_gat = GAT()
print(f"GAT parameters: {sum(p.numel() for p in model_gat.parameters()):,}")
model_gat, hist_gat = run_training(model_gat, 'GAT')


In [ ]:

colors = {'GCN':'#636EFA','GraphSAGE':'#EF553B','GAT':'#00CC96'}
fig3 = make_subplots(rows=1, cols=2, subplot_titles=['Validation Loss', 'Validation Accuracy'])
for name, hist in [('GCN',hist_gcn),('GraphSAGE',hist_sage),('GAT',hist_gat)]:
    ep = list(range(1, len(hist['val_loss'])+1))
    fig3.add_trace(go.Scatter(x=ep, y=hist['val_loss'], name=name, line=dict(color=colors[name],width=2)), row=1, col=1)
    fig3.add_trace(go.Scatter(x=ep, y=hist['val_acc'], name=name, line=dict(color=colors[name],width=2), showlegend=False), row=1, col=2)
fig3.update_layout(title='Baseline GNN Training Curves', height=400, paper_bgcolor='white', plot_bgcolor='white', legend=dict(x=0.75,y=0.95))
fig3.update_xaxes(title_text='Epoch')
fig3.update_yaxes(title_text='BCE Loss', row=1, col=1)
fig3.update_yaxes(title_text='Accuracy', row=1, col=2)
fig3.show()
fig3.write_html(str(FIG_DIR/'fig3_baseline_curves.html'))
print("Figure 3 saved.")


---
## Section 6 · Graph Transformer

The **Graph Transformer** extends graph attention with edge features and Laplacian positional encodings to capture long-range feeder dependencies.


In [ ]:

def compute_laplacian_pe(net, k=8):
    G = build_nx_graph(net)
    L = nx.laplacian_matrix(G, nodelist=sorted(G.nodes())).toarray()
    eigvals, eigvecs = np.linalg.eigh(L)
    pe = eigvecs[:, 1:k+1].astype(np.float32)
    for i in range(pe.shape[1]):
        if pe[:,i].sum() < 0:
            pe[:,i] *= -1
    return torch.tensor(pe, dtype=torch.float32)

LPE_DIM = 8
LPE = compute_laplacian_pe(net, k=LPE_DIM)
print(f"Laplacian PE computed: {LPE.shape}")
for g in pyg_dataset:
    g.lpe = LPE
train_data2 = [pyg_dataset[i] for i in idx[:n_train]]
val_data2 = [pyg_dataset[i] for i in idx[n_train:n_train+n_val]]
test_data2 = [pyg_dataset[i] for i in idx[n_train+n_val:]]
train_loader2 = PyGLoader(train_data2, batch_size=BATCH, shuffle=True)
val_loader2 = PyGLoader(val_data2, batch_size=BATCH, shuffle=False)
test_loader2 = PyGLoader(test_data2, batch_size=BATCH, shuffle=False)


In [ ]:

class GraphTransformer(nn.Module):
    def __init__(self, in_dim=NODE_DIM, lpe_dim=LPE_DIM, edge_dim=EDGE_DIM, hidden=HIDDEN,
                 out_dim=N_SWITCH, n_layers=N_LAYERS, heads=N_HEADS, dropout=DROPOUT):
        super().__init__()
        self.input_proj = nn.Linear(in_dim + lpe_dim, hidden)
        self.edge_proj = nn.Linear(edge_dim, hidden)
        self.convs = nn.ModuleList([TransformerConv(hidden, hidden//heads, heads=heads,
                                                    dropout=dropout, edge_dim=hidden, concat=True)
                                    for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(hidden*2, hidden), nn.GELU(), nn.Dropout(dropout),
                                  nn.Linear(hidden, hidden//2), nn.GELU(), nn.Linear(hidden//2, out_dim))
        self.attn_weights = []
    def forward(self, data, capture_attn=False):
        x, ei, ea, batch = data.x, data.edge_index, data.edge_attr, data.batch
        x = F.gelu(self.input_proj(torch.cat([x, data.lpe], dim=-1)))
        ea = F.gelu(self.edge_proj(ea))
        self.attn_weights = []
        for conv, norm in zip(self.convs, self.norms):
            if capture_attn:
                x_new, (_, aw) = conv(x, ei, ea, return_attention_weights=True)
                self.attn_weights.append(aw.detach().cpu())
            else:
                x_new = conv(x, ei, ea)
            x = norm(F.gelu(x_new) + x)
            x = self.dropout(x)
        return self.head(torch.cat([global_mean_pool(x,batch), global_max_pool(x,batch)], -1))

def run_training_gt(model, model_name, epochs=EPOCHS_BASE, device=DEVICE, lr=LR):
    model = model.to(device)
    opt = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch = CosineAnnealingLR(opt, T_max=epochs, eta_min=lr*0.05)
    crit = nn.BCEWithLogitsLoss()
    history = {'train_loss':[],'val_loss':[],'val_acc':[],'val_f1':[]}
    best_val, best_state = 1e9, None
    for epoch in range(1, epochs+1):
        model.train(); tl = 0
        for batch in train_loader2:
            batch = batch.to(device)
            opt.zero_grad(); out = model(batch); loss = crit(out, batch.y.view(-1, N_SWITCH))
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tl += loss.item() * batch.num_graphs
        tl /= len(train_loader2.dataset)
        vl2, va, vf1, _ = eval_model(model, val_loader2, crit, device)
        sch.step()
        history['train_loss'].append(tl); history['val_loss'].append(vl2)
        history['val_acc'].append(va); history['val_f1'].append(vf1)
        if vl2 < best_val:
            best_val, best_state = vl2, copy.deepcopy(model.state_dict())
        if epoch % 10 == 0:
            print(f"[{model_name}] Ep {epoch:03d} | TL {tl:.4f} VL {vl2:.4f} Acc {va:.4f} F1 {vf1:.4f}")
    model.load_state_dict(best_state)
    torch.save(best_state, str(MODEL_DIR/f'{model_name}.pt'))
    return model, history

model_gt = GraphTransformer()
print(f"Graph Transformer parameters: {sum(p.numel() for p in model_gt.parameters()):,}")
model_gt, hist_gt = run_training_gt(model_gt, 'GraphTransformer')


In [ ]:

sample_batch = next(iter(val_loader2)).to(DEVICE)
model_gt.eval()
_ = model_gt(sample_batch, capture_attn=True)
attn_last = model_gt.attn_weights[-1]
ei = sample_batch.edge_index[:, :37*2].cpu()
A = np.zeros((N_BUSES, N_BUSES))
for k in range(ei.shape[1]):
    u = ei[0,k].item(); v = ei[1,k].item()
    if u < N_BUSES and v < N_BUSES and k < len(attn_last):
        A[u,v] = attn_last[k, 0].item()
fig4 = go.Figure(data=go.Heatmap(z=A, colorscale='Blues', x=[f'B{i+1}' for i in range(N_BUSES)],
                                 y=[f'B{i+1}' for i in range(N_BUSES)], colorbar=dict(title='Attention Weight')))
fig4.update_layout(title='Graph Transformer – Last-Layer Attention Heatmap (Head 0)', height=550, width=600,
                   xaxis=dict(tickfont=dict(size=7)), yaxis=dict(tickfont=dict(size=7), autorange='reversed'))
fig4.show()
fig4.write_html(str(FIG_DIR/'fig4_attention_heatmap.html'))
print("Figure 4 saved.")


---
## Section 7 · Physics-Informed Training

We augment BCE with soft penalties for power balance, voltage limits, and radiality: `L_total = L_BCE + λ1 L_pf + λ2 L_volt + λ3 L_rad`.


In [ ]:

LAMBDA1 = 0.10
LAMBDA2 = 0.30
LAMBDA3 = 0.20
N_RADIAL_CLOSED = 32

def physics_loss(logits, data):
    probs = torch.sigmoid(logits)
    n_closed = probs.sum(dim=-1)
    l_rad = ((n_closed - N_RADIAL_CLOSED)**2).mean()
    nf = data.x; batch = data.batch
    P_net = nf[:,2] - nf[:,4] - nf[:,5] - nf[:,6]
    Vm = nf[:,0]
    Vm_pu = Vm * NF_STD[0].to(Vm.device) + NF_MEAN[0].to(Vm.device)
    l_volt = (F.relu(V_MIN - Vm_pu)**2 + F.relu(Vm_pu - V_MAX)**2).mean()
    P_sum = global_mean_pool(P_net.unsqueeze(-1), batch).squeeze(-1)
    l_pf = (P_sum**2).mean()
    return l_pf, l_volt, l_rad

def run_training_pi(model, model_name, epochs=EPOCHS_BASE, device=DEVICE, lr=LR, use_lpe=False):
    loader_tr = train_loader2 if use_lpe else train_loader
    loader_vl = val_loader2 if use_lpe else val_loader
    model = model.to(device)
    opt = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch = CosineAnnealingLR(opt, T_max=epochs, eta_min=lr*0.05)
    crit = nn.BCEWithLogitsLoss()
    history = {'train_loss':[],'val_loss':[],'val_acc':[],'val_f1':[], 'l_pf':[],'l_volt':[],'l_rad':[]}
    best_val, best_state = 1e9, None
    for epoch in range(1, epochs+1):
        model.train(); tl = ep_pf = ep_volt = ep_rad = 0
        for batch in loader_tr:
            batch = batch.to(device)
            opt.zero_grad()
            logits = model(batch)
            l_bce = crit(logits, batch.y.view(-1,N_SWITCH))
            l_pf, l_volt, l_rad = physics_loss(logits, batch)
            loss = l_bce + LAMBDA1*l_pf + LAMBDA2*l_volt + LAMBDA3*l_rad
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tl += loss.item()*batch.num_graphs; ep_pf += l_pf.item()*batch.num_graphs
            ep_volt += l_volt.item()*batch.num_graphs; ep_rad += l_rad.item()*batch.num_graphs
        N_tr = len(loader_tr.dataset)
        tl/=N_tr; ep_pf/=N_tr; ep_volt/=N_tr; ep_rad/=N_tr
        vl2, va, vf1, _ = eval_model(model, loader_vl, crit, device)
        sch.step()
        history['train_loss'].append(tl); history['val_loss'].append(vl2); history['val_acc'].append(va); history['val_f1'].append(vf1)
        history['l_pf'].append(ep_pf); history['l_volt'].append(ep_volt); history['l_rad'].append(ep_rad)
        if vl2 < best_val:
            best_val, best_state = vl2, copy.deepcopy(model.state_dict())
        if epoch % 10 == 0:
            print(f"[{model_name}] Ep {epoch:03d} | TL {tl:.4f} VL {vl2:.4f} Acc {va:.4f} | l_pf {ep_pf:.4f} l_v {ep_volt:.4f} l_r {ep_rad:.4f}")
    model.load_state_dict(best_state)
    torch.save(best_state, str(MODEL_DIR/f'{model_name}.pt'))
    return model, history

model_pi_gcn = GCN()
print(f"PI-GCN parameters: {sum(p.numel() for p in model_pi_gcn.parameters()):,}")
model_pi_gcn, hist_pi_gcn = run_training_pi(model_pi_gcn, 'PI_GCN', use_lpe=False)
model_pi_gt = GraphTransformer()
print(f"PI-GT parameters: {sum(p.numel() for p in model_pi_gt.parameters()):,}")
model_pi_gt, hist_pi_gt = run_training_pi(model_pi_gt, 'PI_GraphTransformer', use_lpe=True)


In [ ]:

fig5 = make_subplots(rows=1, cols=3, subplot_titles=['Power Flow Violation', 'Voltage Violation', 'Radiality Violation'])
models_pi = {'PI-GCN': hist_pi_gcn, 'PI-GT': hist_pi_gt}
colors_pi = {'PI-GCN':'#AB63FA','PI-GT':'#FFA15A'}
for name, hist in models_pi.items():
    ep = list(range(1, len(hist['l_pf'])+1))
    fig5.add_trace(go.Scatter(x=ep,y=hist['l_pf'], name=name, line=dict(color=colors_pi[name],width=2)), row=1,col=1)
    fig5.add_trace(go.Scatter(x=ep,y=hist['l_volt'], name=name, line=dict(color=colors_pi[name],width=2),showlegend=False), row=1,col=2)
    fig5.add_trace(go.Scatter(x=ep,y=hist['l_rad'], name=name, line=dict(color=colors_pi[name],width=2),showlegend=False), row=1,col=3)
fig5.update_layout(title='Physics Constraint Violations During Training', height=380, paper_bgcolor='white', plot_bgcolor='white')
fig5.update_xaxes(title_text='Epoch')
fig5.show()
fig5.write_html(str(FIG_DIR/'fig5_physics_violations.html'))
print("Figure 5 saved.")


---
## Section 8 · Meta-Learning (MAML)

Model-Agnostic Meta-Learning frames each operating condition as a task and learns an initialization that adapts quickly to new load/DER regimes.


In [ ]:

def classify_task(record):
    solar_frac = record['P_solar_mw'].sum() / (record['P_load_mw'].sum() + 1e-6)
    ev_total = record['P_ev_mw'].sum()
    load_total = record['P_load_mw'].sum()
    load_base = BASE_P.sum()
    vm_min = record['node_f'][:,0].min()
    if solar_frac > 0.30: return 0
    elif ev_total > 0.20: return 1
    elif load_total > 1.30 * load_base: return 2
    elif vm_min < 0.94: return 3
    else: return 4

for rec, g in zip(dataset_raw, pyg_dataset):
    g.task = classify_task(rec)
task_counts = {t: sum(1 for g in pyg_dataset if g.task==t) for t in range(5)}
task_names = {0:'High Solar',1:'High EV',2:'Heavy Load',3:'Voltage Stress',4:'Light/Mixed'}
print("Task distribution:", {task_names[k]:v for k,v in task_counts.items()})


In [ ]:

class MAML:
    def __init__(self, model, lr_inner=1e-3, lr_outer=3e-4, n_inner_steps=5, device=DEVICE):
        self.model = model.to(device)
        self.lr_inner = lr_inner
        self.n_inner_steps = n_inner_steps
        self.outer_opt = AdamW(model.parameters(), lr=lr_outer, weight_decay=1e-4)
        self.device = device
        self.crit = nn.BCEWithLogitsLoss()
    def inner_update(self, support_graphs):
        model_copy = copy.deepcopy(self.model)
        inner_opt = Adam(model_copy.parameters(), lr=self.lr_inner)
        loader = PyGLoader(support_graphs, batch_size=min(16,len(support_graphs)), shuffle=True)
        for _ in range(self.n_inner_steps):
            for batch in loader:
                batch = batch.to(self.device)
                inner_opt.zero_grad()
                loss = self.crit(model_copy(batch), batch.y.view(-1,N_SWITCH))
                loss.backward(); inner_opt.step()
        return model_copy
    def outer_step(self, task_batches):
        self.outer_opt.zero_grad()
        total_loss = 0
        adapted_models = []
        for support, query in task_batches:
            adapted = self.inner_update(support)
            adapted_models.append(adapted)
            q_loader = PyGLoader(query, batch_size=min(16,len(query)), shuffle=False)
            for batch in q_loader:
                batch = batch.to(self.device)
                loss = self.crit(adapted(batch), batch.y.view(-1,N_SWITCH))
                total_loss += loss.item()
        if adapted_models:
            for p_orig, p_adapt in zip(self.model.parameters(), adapted_models[-1].parameters()):
                if p_orig.grad is None:
                    p_orig.grad = (p_orig.data - p_adapt.data) / max(len(task_batches),1)
        nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.outer_opt.step()
        return total_loss / max(len(task_batches),1)

def build_task_episodes(pyg_ds, n_support=10, n_query=20):
    episodes = []
    for t in range(5):
        task_graphs = [g for g in pyg_ds if g.task==t]
        if len(task_graphs) < n_support + n_query:
            continue
        np.random.shuffle(task_graphs)
        episodes.append((task_graphs[:n_support], task_graphs[n_support:n_support+n_query]))
    return episodes

META_EPOCHS = 30
N_SUPPORT = 10
N_QUERY = 20
meta_gcn_base = GCN()
maml = MAML(meta_gcn_base, lr_inner=1e-3, lr_outer=3e-4)
meta_history = {'meta_loss':[], 'val_acc':[], 'val_f1':[]}
crit = nn.BCEWithLogitsLoss()
print("Meta-training (MAML) on GCN …")
for ep in range(1, META_EPOCHS+1):
    episodes = build_task_episodes(train_data, N_SUPPORT, N_QUERY)
    meta_loss = maml.outer_step(episodes)
    vl, va, vf1, _ = eval_model(maml.model, val_loader, crit, DEVICE)
    meta_history['meta_loss'].append(meta_loss); meta_history['val_acc'].append(va); meta_history['val_f1'].append(vf1)
    if ep % 5 == 0:
        print(f"  Meta Ep {ep:03d} | MetaLoss {meta_loss:.4f} ValAcc {va:.4f} ValF1 {vf1:.4f}")
model_meta_gcn = maml.model
torch.save(model_meta_gcn.state_dict(), str(MODEL_DIR/'Meta_GCN.pt'))
print("Meta-GCN trained and saved.")


In [ ]:

class MAML_PI(MAML):
    def inner_update(self, support_graphs):
        model_copy = copy.deepcopy(self.model)
        inner_opt = Adam(model_copy.parameters(), lr=self.lr_inner)
        loader = PyGLoader(support_graphs, batch_size=min(16,len(support_graphs)), shuffle=True)
        for _ in range(self.n_inner_steps):
            for batch in loader:
                batch = batch.to(self.device)
                inner_opt.zero_grad()
                logits = model_copy(batch)
                l_bce = self.crit(logits, batch.y.view(-1,N_SWITCH))
                l_pf, l_volt, l_rad = physics_loss(logits, batch)
                loss = l_bce + LAMBDA1*l_pf + LAMBDA2*l_volt + LAMBDA3*l_rad
                loss.backward(); inner_opt.step()
        return model_copy

meta_pi_gt_base = GraphTransformer()
maml_pi_gt = MAML_PI(meta_pi_gt_base, lr_inner=1e-3, lr_outer=3e-4)
meta_pi_gt_hist = {'meta_loss':[],'val_acc':[],'val_f1':[]}
print("Meta-training PI Graph Transformer …")
for ep in range(1, META_EPOCHS+1):
    episodes = build_task_episodes(train_data2, N_SUPPORT, N_QUERY)
    meta_loss = maml_pi_gt.outer_step(episodes)
    vl, va, vf1, _ = eval_model(maml_pi_gt.model, val_loader2, crit, DEVICE)
    meta_pi_gt_hist['meta_loss'].append(meta_loss); meta_pi_gt_hist['val_acc'].append(va); meta_pi_gt_hist['val_f1'].append(vf1)
    if ep % 5 == 0:
        print(f"  Meta-PI-GT Ep {ep:03d} | MetaLoss {meta_loss:.4f} ValAcc {va:.4f} ValF1 {vf1:.4f}")
model_meta_pi_gt = maml_pi_gt.model
torch.save(model_meta_pi_gt.state_dict(), str(MODEL_DIR/'Meta_PI_GraphTransformer.pt'))
print("Meta-PI-GT trained and saved.")


In [ ]:

def few_shot_accuracy(meta_model, task_graphs, k_shots, n_eval=50, device=DEVICE):
    support = task_graphs[:k_shots]
    query = task_graphs[k_shots:k_shots+n_eval]
    if len(query) == 0: return 0.0
    model_copy = copy.deepcopy(meta_model).to(device)
    if k_shots > 0:
        inner_opt = Adam(model_copy.parameters(), lr=1e-3)
        loader = PyGLoader(support, batch_size=k_shots, shuffle=False)
        crit_l = nn.BCEWithLogitsLoss()
        for _ in range(10):
            for batch in loader:
                batch = batch.to(device)
                inner_opt.zero_grad()
                loss = crit_l(model_copy(batch), batch.y.view(-1,N_SWITCH))
                loss.backward(); inner_opt.step()
    model_copy.eval()
    q_loader = PyGLoader(query, batch_size=n_eval, shuffle=False)
    all_p, all_l = [], []
    with torch.no_grad():
        for batch in q_loader:
            batch = batch.to(device)
            all_p.append(torch.sigmoid(model_copy(batch)).cpu().numpy())
            all_l.append(batch.y.view(-1,N_SWITCH).cpu().numpy())
    P = np.vstack(all_p); L = np.vstack(all_l)
    return accuracy_score(L.flatten(),(P>0.5).astype(int).flatten())

K_SHOTS = [0, 1, 5, 10, 20]
adapt_results = {}
for task_id in range(5):
    tg = [g for g in pyg_dataset if g.task==task_id]
    if len(tg) < 70: continue
    np.random.shuffle(tg)
    row_meta, row_vanilla = [], []
    for k in K_SHOTS:
        row_meta.append(few_shot_accuracy(model_meta_gcn, tg[:60], k_shots=k))
        row_vanilla.append(few_shot_accuracy(model_gcn, tg[:60], k_shots=k))
    adapt_results[task_names[task_id]] = {'meta':row_meta, 'vanilla':row_vanilla}
    print(f"{task_names[task_id]:15s} | meta: {row_meta} | vanilla: {row_vanilla}")

fig6 = make_subplots(rows=1, cols=min(5,len(adapt_results)), subplot_titles=list(adapt_results.keys()))
for col, (tname, vals) in enumerate(adapt_results.items(), 1):
    fig6.add_trace(go.Scatter(x=K_SHOTS, y=vals['meta'], name='Meta-GCN', line=dict(color='#636EFA',width=2), marker=dict(size=7), showlegend=(col==1)), row=1, col=col)
    fig6.add_trace(go.Scatter(x=K_SHOTS, y=vals['vanilla'], name='GCN (no meta)', line=dict(color='#EF553B',width=2,dash='dash'), marker=dict(size=7), showlegend=(col==1)), row=1, col=col)
fig6.update_layout(title='Few-Shot Adaptation Accuracy by Task', height=380, paper_bgcolor='white', plot_bgcolor='white')
fig6.update_xaxes(title_text='K shots')
fig6.update_yaxes(title_text='Accuracy', range=[0,1])
fig6.show()
fig6.write_html(str(FIG_DIR/'fig6_adaptation_curves.html'))
print("Figure 6 saved.")


---
## Section 9 · Ablation Study

Six model variants are trained and evaluated on the held-out test set: GCN, PI-GCN, Graph Transformer, PI-GT, Meta-GCN, and Meta-PI-GT.


In [ ]:

def full_evaluate(model, loader, device=DEVICE):
    model.eval()
    all_p, all_l = [], []
    total_loss = 0
    t0 = time.time()
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch)
            loss = crit(logits, batch.y.view(-1,N_SWITCH))
            total_loss += loss.item()*batch.num_graphs
            all_p.append(torch.sigmoid(logits).cpu().numpy())
            all_l.append(batch.y.view(-1,N_SWITCH).cpu().numpy())
    elapsed = time.time()-t0
    P = np.vstack(all_p); L = np.vstack(all_l); Pb = (P>0.5).astype(int)
    acc = accuracy_score(L.flatten(), Pb.flatten())
    prec = precision_score(L.flatten(), Pb.flatten(), zero_division=0)
    rec = recall_score(L.flatten(), Pb.flatten(), zero_division=0)
    f1 = f1_score(L.flatten(), Pb.flatten(), zero_division=0)
    try: auc = roc_auc_score(L.flatten(), P.flatten())
    except Exception: auc = 0.5
    n_closed = Pb.sum(axis=1)
    rad_sat = (n_closed == N_RADIAL_CLOSED).mean()
    exact_match = (Pb == L).all(axis=1).mean()
    inf_speed = elapsed / len(loader.dataset) * 1000
    return dict(loss=total_loss/len(loader.dataset), acc=acc, prec=prec, rec=rec, f1=f1,
                auc=auc, rad_sat=rad_sat, exact_match=exact_match, inf_ms=inf_speed,
                preds=P, labels=L)

MODELS = {
    'A-GCN': (model_gcn, test_loader),
    'B-PI-GCN': (model_pi_gcn, test_loader),
    'C-GT': (model_gt, test_loader2),
    'D-PI-GT': (model_pi_gt, test_loader2),
    'E-Meta-GCN': (model_meta_gcn, test_loader),
    'F-Meta-PI-GT': (model_meta_pi_gt, test_loader2),
}
ablation = {}
for name, (mdl, loader) in MODELS.items():
    print(f"Evaluating {name} …", end=' ')
    ablation[name] = full_evaluate(mdl, loader)
    r = ablation[name]
    print(f"Acc={r['acc']:.4f}  F1={r['f1']:.4f}  AUC={r['auc']:.4f}  Rad={r['rad_sat']:.3f}  Exact={r['exact_match']:.3f}  {r['inf_ms']:.2f}ms/g")


In [ ]:

rows = []
for name, res in ablation.items():
    rows.append({'Model': name, 'Accuracy': f"{res['acc']:.4f}", 'Precision': f"{res['prec']:.4f}",
                 'Recall': f"{res['rec']:.4f}", 'F1': f"{res['f1']:.4f}", 'AUC': f"{res['auc']:.4f}",
                 'Radiality': f"{res['rad_sat']:.3f}", 'ExactMatch': f"{res['exact_match']:.3f}",
                 'Inf (ms/g)': f"{res['inf_ms']:.2f}"})
df_abl = pd.DataFrame(rows).set_index('Model')
print("\n=== ABLATION STUDY RESULTS ===")
print(df_abl.to_string())
df_abl.to_csv(str(DATA_DIR/'ablation_results.csv'))
metrics = ['Accuracy','F1','AUC','Radiality','ExactMatch']
model_names = list(ablation.keys())
palette = px.colors.qualitative.Plotly
fig7 = make_subplots(rows=2, cols=3, subplot_titles=metrics+['Inference Speed (ms/graph)'])
positions = [(1,1),(1,2),(1,3),(2,1),(2,2),(2,3)]
for (r,c), met in zip(positions[:5], metrics):
    vals = [float(df_abl.loc[m,met]) for m in model_names]
    fig7.add_trace(go.Bar(x=model_names, y=vals, name=met, marker_color=palette[:len(model_names)], showlegend=False), row=r, col=c)
    fig7.update_yaxes(title_text=met, row=r, col=c)
spd = [float(df_abl.loc[m,'Inf (ms/g)']) for m in model_names]
fig7.add_trace(go.Bar(x=model_names, y=spd, name='Inference', marker_color=palette[:len(model_names)], showlegend=False), row=2, col=3)
fig7.update_yaxes(title_text='ms/graph', row=2, col=3)
fig7.update_layout(title='Ablation Study – All Model Variants', height=600, paper_bgcolor='white', plot_bgcolor='white')
fig7.show()
fig7.write_html(str(FIG_DIR/'fig7_ablation_bars.html'))
print("Figure 7 saved.")


---
## Section 10 · Out-of-Distribution (OOD) Testing

We create five out-of-distribution scenario families never seen during training: extreme loading, extreme solar, multi-DER outage, line outage, and storm conditions.


In [ ]:

N_OOD = 500

def generate_ood_scenarios(n=N_OOD, seed=999):
    rng = np.random.default_rng(seed)
    ood = []
    types = ['extreme_load','extreme_solar','der_outage','line_outage','storm']
    per_t = n // len(types)
    for otype in types:
        for _ in range(per_t):
            s = {'P_load_mw': BASE_P.copy(), 'Q_load_mvar': BASE_Q.copy(),
                 'P_solar_mw': np.zeros(N_BUSES), 'P_wind_mw': np.zeros(N_BUSES),
                 'P_batt_mw': np.zeros(N_BUSES), 'SoC': np.zeros(N_BUSES),
                 'P_ev_mw': np.zeros(N_BUSES), 'ood_type': otype}
            if otype == 'extreme_load':
                mult = rng.uniform(1.8, 2.0, size=len(BASE_P)); s['P_load_mw'] *= mult; s['Q_load_mvar'] *= mult
            elif otype == 'extreme_solar':
                for b in SOLAR_BUSES: s['P_solar_mw'][b] = rng.uniform(0.6, 1.0)
            elif otype == 'line_outage':
                s['outage_line'] = int(rng.integers(0, 32))
            elif otype == 'storm':
                for b in WIND_BUSES: s['P_wind_mw'][b] = rng.uniform(0.12, 0.18)
                mult = rng.uniform(1.3, 1.5, size=len(BASE_P)); s['P_load_mw'] *= mult; s['Q_load_mvar'] *= mult
            ood.append(s)
    return ood

ood_scenarios = generate_ood_scenarios()
print(f"OOD scenarios: {len(ood_scenarios)}")
ood_graphs, ood_types = [], []
for s in ood_scenarios:
    try:
        net_s = apply_scenario(net, s)
        if 'outage_line' in s:
            net_s.line.at[s['outage_line'], 'in_service'] = False
        set_switch_config(net_s, 0)
        pp.runpp(net_s, algorithm='nr', numba=False, max_iteration=50)
        if not net_s.converged: continue
        node_f = extract_bus_features(net_s)
        opt_c, _ = find_optimal_config(net_s)
        net_opt = copy.deepcopy(net_s); set_switch_config(net_opt, opt_c); pp.runpp(net_opt, algorithm='nr', numba=False)
        edge_f = extract_line_features(net_opt)
        sw_l = np.ones(37, dtype=np.float32)
        for bit, li in enumerate(TIE_LINE_IDX): sw_l[li] = float((opt_c >> bit) & 1)
        nf = (torch.tensor(node_f, dtype=torch.float32) - NF_MEAN) / NF_STD
        ea = torch.tensor(np.concatenate([edge_f,edge_f],0), dtype=torch.float32)
        g = Data(x=nf, edge_index=EDGE_INDEX.clone(), edge_attr=ea, y=torch.tensor(sw_l,dtype=torch.float32), lpe=LPE)
        g.ood_type = s['ood_type']
        ood_graphs.append(g); ood_types.append(s['ood_type'])
    except Exception:
        continue
print(f"Valid OOD graphs: {len(ood_graphs)}")
ood_loader = PyGLoader(ood_graphs, batch_size=BATCH, shuffle=False)


In [ ]:

ood_type_list = list(set(ood_types))
def eval_ood_by_type(model, ood_graphs, ood_types, device=DEVICE):
    model.eval()
    results = {t:{'acc':[],'f1':[]} for t in ood_type_list}
    with torch.no_grad():
        for g, otype in zip(ood_graphs, ood_types):
            g2 = g.clone().to(device)
            g2.batch = torch.zeros(g2.num_nodes, dtype=torch.long, device=device)
            logits = model(g2)
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
            labels = g2.y.cpu().numpy().flatten()
            bp = (probs>0.5).astype(int)
            results[otype]['acc'].append(accuracy_score(labels, bp))
            results[otype]['f1'].append(f1_score(labels, bp, zero_division=0))
    return {t: {'acc': np.mean(v['acc']), 'f1': np.mean(v['f1'])} for t, v in results.items()}

print("OOD evaluation …")
ood_results = {}
for name, (mdl, _) in MODELS.items():
    ood_results[name] = eval_ood_by_type(mdl, ood_graphs, ood_types)
    avg_acc = np.mean([v['acc'] for v in ood_results[name].values()])
    avg_f1 = np.mean([v['f1'] for v in ood_results[name].values()])
    print(f"  {name:18s} | OOD Acc {avg_acc:.4f}  OOD F1 {avg_f1:.4f}")
fig8 = go.Figure()
for name in MODELS.keys():
    accs = [ood_results[name].get(t,{}).get('acc',0) for t in sorted(ood_type_list)]
    fig8.add_trace(go.Bar(name=name, x=sorted(ood_type_list), y=accs))
fig8.update_layout(title='OOD Accuracy by Scenario Type', barmode='group', height=420,
                   yaxis=dict(title='Accuracy', range=[0,1]), paper_bgcolor='white', plot_bgcolor='white')
fig8.show(); fig8.write_html(str(FIG_DIR/'fig8_ood_accuracy.html'))
print("Figure 8 saved.")


---
## Section 11 · Comprehensive Visualisation Dashboard

In [ ]:

sample_sc = scenarios[0]
net_s = apply_scenario(net, sample_sc)
set_switch_config(net_s, 0); pp.runpp(net_s, algorithm='nr', numba=False)
v_before = net_s.res_bus.vm_pu.values.copy()
opt_c, _ = find_optimal_config(net_s)
set_switch_config(net_s, opt_c); pp.runpp(net_s, algorithm='nr', numba=False)
v_after = net_s.res_bus.vm_pu.values.copy()
buses = list(range(1, N_BUSES+1))
fig9 = go.Figure()
fig9.add_trace(go.Scatter(x=buses, y=v_before, mode='lines+markers', name='Before Reconfiguration', line=dict(color='tomato', width=2), marker=dict(size=6)))
fig9.add_trace(go.Scatter(x=buses, y=v_after, mode='lines+markers', name='After Reconfiguration', line=dict(color='steelblue', width=2), marker=dict(size=6)))
fig9.add_hline(y=V_MIN, line_dash='dash', line_color='red', annotation_text='V_min = 0.95 pu')
fig9.add_hline(y=V_MAX, line_dash='dash', line_color='green', annotation_text='V_max = 1.05 pu')
fig9.update_layout(title='Voltage Profile – Before vs After Reconfiguration (Scenario 0)', xaxis_title='Bus Number', yaxis_title='Voltage (pu)', height=420, paper_bgcolor='white', plot_bgcolor='white')
fig9.show(); fig9.write_html(str(FIG_DIR/'fig9_voltage_profile.html'))
print("Figure 9 saved.")


In [ ]:

radar_metrics = ['Accuracy','F1','AUC','Radiality','ExactMatch']
fig10 = go.Figure()
for name in ablation:
    vals = [float(df_abl.loc[name, m]) for m in radar_metrics]
    fig10.add_trace(go.Scatterpolar(r=vals + [vals[0]], theta=radar_metrics + [radar_metrics[0]], name=name, fill='toself', opacity=0.5))
fig10.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0,1])), title='Radar Chart – Model Comparison', height=520, paper_bgcolor='white')
fig10.show(); fig10.write_html(str(FIG_DIR/'fig10_radar.html'))
print("Figure 10 saved.")

freq_matrix = np.zeros((len(MODELS), N_SWITCH))
for i, (name, res) in enumerate(ablation.items()):
    freq_matrix[i] = (res['preds'] > 0.5).astype(int).mean(axis=0)
fig11 = go.Figure(data=go.Heatmap(z=freq_matrix, x=[f'SW{j+1}' for j in range(N_SWITCH)], y=list(ablation.keys()), colorscale='Viridis', colorbar=dict(title='Closure Freq.')))
fig11.update_layout(title='Switch Closure Frequency Across Test Set', xaxis_title='Switch Index', yaxis_title='Model', height=350, paper_bgcolor='white')
fig11.show(); fig11.write_html(str(FIG_DIR/'fig11_switch_frequency.html'))
print("Figure 11 saved.")

fig12 = go.Figure()
for name, hist in [('GCN',hist_gcn),('GT',hist_gt),('PI-GCN',hist_pi_gcn),('PI-GT',hist_pi_gt)]:
    fig12.add_trace(go.Scatter(x=list(range(1,len(hist['train_loss'])+1)), y=hist['train_loss'], name=f'{name} Train', mode='lines', line=dict(width=1.5)))
fig12.update_layout(title='Training Loss Comparison – All Models', xaxis_title='Epoch', yaxis_title='BCE Loss', height=420, paper_bgcolor='white', plot_bgcolor='white')
fig12.show(); fig12.write_html(str(FIG_DIR/'fig12_loss_curves.html'))
print("Figure 12 saved.")


---
## Section 12 · Statistical Testing

We apply Wilcoxon signed-rank tests and paired t-tests to evaluate whether performance differences between model pairs are statistically significant.


In [ ]:

def per_graph_f1(res):
    P = (res['preds'] > 0.5).astype(int)
    L = res['labels'].astype(int)
    return np.array([f1_score(l_row, p_row, zero_division=0) for p_row, l_row in zip(P, L)])

per_f1 = {name: per_graph_f1(res) for name,res in ablation.items()}
comparisons = [('A-GCN','C-GT','GCN vs GT'), ('C-GT','D-PI-GT','GT vs PI-GT'),
               ('D-PI-GT','F-Meta-PI-GT','PI-GT vs Meta-PI-GT'),
               ('A-GCN','F-Meta-PI-GT','GCN vs Meta-PI-GT (best)')]
stat_rows = []
print(f"{'Comparison':30s}  {'t-stat':>8}  {'t-pval':>8}  {'W-stat':>8}  {'W-pval':>8}  {'EffSize':>8}")
print('-'*80)
for m1, m2, label in comparisons:
    a, b = per_f1[m1], per_f1[m2]
    n = min(len(a), len(b)); a, b = a[:n], b[:n]
    t_stat, t_pval = ttest_rel(a, b)
    try: w_stat, w_pval = wilcoxon(a-b, zero_method='wilcox')
    except Exception: w_stat, w_pval = 0, 1.0
    diff = a - b
    effect = diff.mean() / (diff.std() + 1e-9)
    print(f"{label:30s}  {t_stat:8.3f}  {t_pval:8.4f}  {w_stat:8.1f}  {w_pval:8.4f}  {effect:8.4f}")
    stat_rows.append({'Comparison':label, 't-stat':round(t_stat,4), 't-pval':round(t_pval,4),
                      'W-stat':round(w_stat,2), 'W-pval':round(w_pval,4), "Cohen's d":round(effect,4),
                      'Significant (α=0.05)': 'Yes' if t_pval<0.05 else 'No'})
df_stats = pd.DataFrame(stat_rows)
df_stats.to_csv(str(DATA_DIR/'statistical_tests.csv'), index=False)
print("\nStatistical test results saved.")

def bootstrap_ci(scores, n_boot=2000, ci=0.95):
    rng = np.random.default_rng(SEED)
    boot = [rng.choice(scores, len(scores), replace=True).mean() for _ in range(n_boot)]
    return np.percentile(boot, (1-ci)/2*100), np.percentile(boot, (1+ci)/2*100)

print("\n95% Bootstrap CIs for Test-Set F1:")
for name, f1s in per_f1.items():
    lo, hi = bootstrap_ci(f1s)
    print(f"  {name:20s}: {f1s.mean():.4f}  [{lo:.4f}, {hi:.4f}]")


---
## Section 13 · Research Analysis

### 13.1 Discussion

This study compared six graph-based machine learning architectures for real-time distribution network reconfiguration on the IEEE 33-bus system.

### 13.2 Automated Observations Generator

In [ ]:

best_model = max(ablation, key=lambda n: ablation[n]['f1'])
worst_model = min(ablation, key=lambda n: ablation[n]['f1'])
best_rad = max(ablation, key=lambda n: ablation[n]['rad_sat'])
fastest = min(ablation, key=lambda n: ablation[n]['inf_ms'])
best_ood = max(ablation, key=lambda n: np.mean([ood_results[n].get(t,{}).get('acc',0) for t in ood_type_list]))
obs_text = f"""
=== AUTOMATED RESEARCH OBSERVATIONS ===

1. BEST OVERALL MODEL
   → {best_model} achieves the highest F1 = {ablation[best_model]['f1']:.4f}
     and AUC = {ablation[best_model]['auc']:.4f} on the test set.

2. PHYSICS REGULARISATION EFFECT
   GCN F1      = {ablation['A-GCN']['f1']:.4f}
   PI-GCN F1   = {ablation['B-PI-GCN']['f1']:.4f}
   Delta F1    = {ablation['B-PI-GCN']['f1']-ablation['A-GCN']['f1']:+.4f}

3. TRANSFORMER ADVANTAGE
   GCN F1 = {ablation['A-GCN']['f1']:.4f}   GT F1 = {ablation['C-GT']['f1']:.4f}

4. META-LEARNING GENERALISATION
   GCN OOD Acc      = {np.mean([ood_results['A-GCN'].get(t,{}).get('acc',0) for t in ood_type_list]):.4f}
   Meta-GCN OOD Acc = {np.mean([ood_results['E-Meta-GCN'].get(t,{}).get('acc',0) for t in ood_type_list]):.4f}

5. RADIALITY CONSTRAINT SATISFACTION
   Best: {best_rad} → {ablation[best_rad]['rad_sat']:.3f}

6. INFERENCE SPEED
   Fastest model: {fastest} at {ablation[fastest]['inf_ms']:.2f} ms/graph

7. BEST OOD MODEL: {best_ood}

=== LIMITATIONS ===
1. Heuristic label generation, not true global optimum.
2. Linearised physics penalties, not full differentiable AC power flow.
3. Fixed topology.
4. DER forecast uncertainty not modelled probabilistically.
5. Battery degradation and multi-period operation not considered.
"""
print(obs_text)
with open(str(DATA_DIR/'research_observations.txt'),'w') as f:
    f.write(obs_text)


---
## Section 14 · Final Recommendation & Publication-Quality Conclusion

In [ ]:

W = {'F1':0.30, 'AUC':0.10, 'Radiality':0.20, 'ExactMatch':0.15, 'OOD_Acc':0.20, 'Speed_norm':0.05}
all_speeds = np.array([ablation[n]['inf_ms'] for n in ablation])
spd_norm = 1 - (all_speeds - all_speeds.min()) / (all_speeds.max()-all_speeds.min()+1e-9)
ranking_rows = []
for i, name in enumerate(ablation):
    res = ablation[name]
    ood_avg = np.mean([ood_results[name].get(t,{}).get('acc',0) for t in ood_type_list])
    composite = W['F1']*res['f1'] + W['AUC']*res['auc'] + W['Radiality']*res['rad_sat'] + W['ExactMatch']*res['exact_match'] + W['OOD_Acc']*ood_avg + W['Speed_norm']*spd_norm[i]
    ranking_rows.append({'Model':name, 'F1':round(res['f1'],4), 'AUC':round(res['auc'],4),
                         'Radiality':round(res['rad_sat'],3), 'ExactMatch':round(res['exact_match'],3),
                         'OOD Acc':round(ood_avg,4), 'Inf (ms/g)':round(res['inf_ms'],2),
                         'COMPOSITE':round(composite,4)})
df_rank = pd.DataFrame(ranking_rows).sort_values('COMPOSITE', ascending=False)
df_rank['Rank'] = range(1, len(df_rank)+1)
df_rank = df_rank.set_index('Rank')
print("\n=== FINAL MODEL RANKING ===")
print(df_rank.to_string())
df_rank.to_csv(str(DATA_DIR/'final_ranking.csv'))


In [ ]:

df_r = df_rank.reset_index()
colors_rank = ['gold','silver','#CD7F32','#636EFA','#EF553B','#00CC96']
fig14 = go.Figure()
fig14.add_trace(go.Bar(x=df_r['Model'], y=df_r['COMPOSITE'], marker_color=colors_rank[:len(df_r)],
                       text=[f"#{r}" for r in df_r['Rank']], textposition='outside',
                       hovertext=[f"{row['Model']}<br>F1={row['F1']}<br>AUC={row['AUC']}<br>Rad={row['Radiality']}<br>OOD={row['OOD Acc']}<br>Speed={row['Inf (ms/g)']}ms" for _, row in df_r.iterrows()],
                       hoverinfo='text'))
fig14.update_layout(title='Final Model Ranking – Weighted Composite Score<br><sup>Weights: F1×0.30  Radiality×0.20  OOD×0.20  ExactMatch×0.15  AUC×0.10  Speed×0.05</sup>',
                    yaxis=dict(title='Composite Score', range=[0,1]), xaxis_title='Model', height=480,
                    paper_bgcolor='white', plot_bgcolor='white', font=dict(size=13))
fig14.show(); fig14.write_html(str(FIG_DIR/'fig14_final_ranking.html'))
print("Figure 14 saved.")


In [ ]:

top_model = df_rank.iloc[0]
second = df_rank.iloc[1]
conclusion = f"""
================================================================================
CONCLUSION
================================================================================

This study systematically evaluated six graph-based machine learning architectures
for real-time Distribution Network Reconfiguration (DNR) on the IEEE 33-bus system,
with 10 000 synthetic operating scenarios encompassing load variability, solar PV,
wind generation, battery storage, and EV charging demand.

RECOMMENDED ARCHITECTURE: {top_model['Model']}
Composite Score : {top_model['COMPOSITE']:.4f}
F1 Score        : {top_model['F1']:.4f}
AUC             : {top_model['AUC']:.4f}
Radiality Sat.  : {top_model['Radiality']:.3f}
OOD Accuracy    : {top_model['OOD Acc']:.4f}
Inference Speed : {top_model['Inf (ms/g)']:.2f} ms/graph

RUNNER-UP: {second['Model']} (Composite: {second['COMPOSITE']:.4f})

PRACTICAL RECOMMENDATION FOR UTILITIES:
Deploy the {top_model['Model']} as the primary DNR decision engine within the
DMS inference pipeline. Retrain periodically using newly collected operational data.
Use the meta-learning capability for rapid adaptation when new DER assets are commissioned.
================================================================================
"""
print(conclusion)
with open(str(DATA_DIR/'conclusion.txt'),'w') as f:
    f.write(conclusion)


---
## Summary of All Outputs

| File | Description |
|------|-------------|
| `data/scenarios.npy` | Cached synthetic scenarios for kernel-restart recovery |
| `data/dataset_raw.npy` | Cached labeled dataset (10 000 scenarios) used to resume from PyG conversion/training |
| `data/ablation_results.csv` | Test-set metrics for all 6 models |
| `data/statistical_tests.csv` | Wilcoxon / t-test results |
| `data/final_ranking.csv` | Composite ranking |
| `data/research_observations.txt` | Auto-generated observations |
| `data/conclusion.txt` | Publication-quality conclusion |
| `models/*.pt` | Saved model weights |
| `figures/fig*.html` | Interactive Plotly figures |
